# 🧠 Aula 06: Introdução à Visão Computacional e Redes Convolucionais (CNNs)

Bem-vindo(a) à **Aula 06** do curso de Redes Neurais Profundas da Graduação da Faculdade Infnet!

Nesta aula, ingressamos no universo da **Visão Computacional**:
compreenderemos como imagens digitais são representadas em tensores 2D/3D, investigaremos a falha estrutural do estiramento (*flattening*) de imagens em redes MLP, aprenderemos a operação matemática da **Convolução 2D**, o papel dos **Kernels** como detectores automáticos de bordas e texturas, os hiperparâmetros de **Padding** e **Stride**, o papel das camadas de **Pooling** (Max vs Average) na invariância translacional e construiremos nossa primeira **Rede Neural Convolucional (CNN)** no PyTorch para classificar dígitos do MNIST.

---

### 🎯 Objetivos de Aprendizagem
1. **A Imagem como Tensor**: Interpretar matrizes de pixels com valores no intervalo [0, 255] e tensores normalizados no PyTorch.
2. **Diagnosticar a Falha do Flattening**: Identificar por que achatar imagens em vetores 1D quebra a localidade espacial e causa explosão de parâmetros.
3. **A Operação de Convolução 2D**: Compreender o produto interno deslizante entre filtros (*kernels*) e a matriz de entrada.
4. **Kernels como Extratores de Features**: Analisar como matrizes pequenas (3x3 ou 5x5) detectam bordas horizontais, verticais e relevos.
5. **Hiperparâmetros Convolucionais**: Calcular a dimensão de saída $O = \lfloor\frac{W - K + 2P}{S}\rfloor + 1$ variando Kernel ($K$), Padding ($P$) e Stride ($S$).
6. **Subamostragem com Pooling**: Explicar a função do *Max Pooling* na compressão dimensional e na criação de invariância espacial.
7. **Construir e Treinar a MNISTCNN**: Implementar e treinar no PyTorch com `nn.Conv2d`, `nn.MaxPool2d` e `nn.Linear`, comparando o ganho de acurácia frente à MLP.

---
### 1. Importação de Bibliotecas & Configuração de Dispositivo (CPU/GPU)


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms

import matplotlib.pyplot as plt
import numpy as np

# Configuração de Seed para Reproducilidade
torch.manual_seed(42)

# Configuração de Dispositivo (GPU no Colab se disponível)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo em uso: {device}")


---
### 2. Carregamento do Dataset MNIST via Torchvision & DataLoader
O dataset MNIST é composto por $70.000$ imagens em escala de cinza de dígitos manuscritos ($0$ a $9$) com dimensão $28 	imes 28$ pixels.


In [ ]:
# Transformações: Converter imagem PIL para Tensor PyTorch com valores normalizados em [0, 1]
transform = transforms.Compose([
    transforms.ToTensor()
])

# Download e carregamento dos conjuntos de Treino e Teste
train_dataset = torchvision.datasets.MNIST(
    root='./data', 
    train=True, 
    download=True, 
    transform=transform
)

test_dataset = torchvision.datasets.MNIST(
    root='./data', 
    train=False, 
    download=True, 
    transform=transform
)

# DataLoaders para iteração em minibatches
batch_size = 64

train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=False)

print(f"Total de imagens no Treino: {len(train_dataset)}")
print(f"Total de imagens no Teste: {len(test_dataset)}")
print(f"Número de batches por época (Treino): {len(train_loader)}")


---
### 3. Inspeção Visual e Formato do Tensor
Vamos visualizar algumas amostras e verificar o formato dos tensores.


In [ ]:
# Extrair 1 batch de dados
data_iter = iter(train_loader)
images, labels = next(data_iter)

print(f"Formato do Tensor de Imagens (Batch, Canais, Altura, Largura): {images.shape}")
print(f"Formato dos Rótulos (Labels): {labels.shape}")

# Plot de 8 amostras
fig, axes = plt.subplots(1, 8, figsize=(14, 2))
for i in range(8):
    axes[i].imshow(images[i].squeeze().numpy(), cmap='gray')
    axes[i].set_title(f"Label: {labels[i].item()}")
    axes[i].axis('off')
plt.tight_layout()
plt.show()


---
## 1. O Modelo MLP para Imagens e Suas Limitações Estruturais

Treinamos primeiro uma rede MLP tradicional para estabelecer o baseline e evidenciar as fraquezas do estiramento (*flattening*).

---
### 4. Construção do Modelo MLP (`nn.Flatten` + Camadas Densas)
Em uma MLP tradicional, não usamos convoluções. A matriz $28 	imes 28$ de cada imagem é "estirada" (*flattened*) para um único vetor $1D$ de $28 	imes 28 = 784$ posições.

$$\mathbf{x} \in \mathbb{R}^{1 \times 28 \times 28} \xrightarrow{\text{Flatten}} \mathbf{x}_{1D} \in \mathbb{R}^{784}$$


In [ ]:
class MNISTMLP(nn.Module):
    def __init__(self):
        super(MNISTMLP, self).__init__()
        
        # Converte o tensor (B, 1, 28, 28) -> (B, 784)
        self.flatten = nn.Flatten()
        
        # Camadas Densas (Fully Connected / Linear)
        self.classifier = nn.Sequential(
            nn.Linear(784, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 10)  # 10 classes (dígitos 0 a 9)
        )
        
    def forward(self, x):
        x = self.flatten(x)
        logits = self.classifier(x)
        return logits

# Instanciar modelo
model_mlp = MNISTMLP().to(device)
print(model_mlp)

# Contagem de parâmetros treináveis
total_params = sum(p.numel() for p in model_mlp.parameters() if p.requires_grad)
print(f"Total de parâmetros treináveis da MLP: {total_params:,}")


---
### 5. Configuração da Função de Perda & Otimizador


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_mlp.parameters(), lr=0.001)


---
### 6. Loop de Treinamento e Validação da MLP


In [ ]:
epochs = 5

train_losses = []
train_accs = []

for epoch in range(epochs):
    model_mlp.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        # Forward pass
        outputs = model_mlp(images)
        loss = criterion(outputs, labels)
        
        # Backward pass e Otimização
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # Métricas
        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
    epoch_loss = running_loss / total
    epoch_acc = (correct / total) * 100
    
    train_losses.append(epoch_loss)
    train_accs.append(epoch_acc)
    
    print(f"Época [{epoch+1}/{epochs}] | Loss: {epoch_loss:.4f} | Acurácia Treino: {epoch_acc:.2f}%")


---
### 7. Avaliação no Conjunto de Teste


In [ ]:
model_mlp.eval()
test_loss = 0.0
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model_mlp(images)
        loss = criterion(outputs, labels)
        
        test_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

final_test_acc = (correct / total) * 100
print(f"📊 Acurácia Final da MLP no Conjunto de Teste: {final_test_acc:.2f}%")


---
### 8. Conclusão da Parte 1 & Limitações da MLP em Imagens
- A MLP atinge boa acurácia em dígitos simples de baixa resolução ($28 	imes 28$).
- **Porém**, para estirar a matriz 2D, ela perdeu completamente a noção de vizinhança espacial entre os pixels superiores e inferiores.
- Na Parte 2, veremos como utilizar **Camadas Convolucionais (`nn.Conv2d`)** para preservar a estrutura espacial e alcançar resultados ainda superiores com maior eficiência!


---
## 2. A Solução Convolucional: Arquitetura `MNISTCNN`

Em vez de achatar a matriz 2D logo na entrada, utilizamos camadas `nn.Conv2d` que preservam a vizinhança espacial entre pixels e compartilham pesos (*parameter sharing*).

---
### 3. Arquitetura da Rede Neural Convolucional (`MNISTCNN`)
A arquitetura convolucional preserva a grade $2D$ da imagem $(1, 28, 28)$ e aplica:
1. **Bloco Convolucional 1:** `Conv2d(1, 16, kernel_size=3, padding=1)` + `ReLU` + `MaxPool2d(2, 2)` $\rightarrow$ Saída $(16, 14, 14)$
2. **Bloco Convolucional 2:** `Conv2d(16, 32, kernel_size=3, padding=1)` + `ReLU` + `MaxPool2d(2, 2)` $\rightarrow$ Saída $(32, 7, 7)$
3. **Classificador Denso:** `Flatten()` + `Linear(32 * 7 * 7, 128)` + `ReLU` + `Linear(128, 10)`


In [ ]:
class MNISTCNN(nn.Module):
    def __init__(self):
        super(MNISTCNN, self).__init__()
        
        # Bloco Convolucional 1: (1, 28, 28) -> (16, 14, 14)
        self.conv_block1 = nn.Sequential(
            nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        
        # Bloco Convolucional 2: (16, 14, 14) -> (32, 7, 7)
        self.conv_block2 = nn.Sequential(
            nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        
        # Classificador Linear Final
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 7 * 7, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )
        
    def forward(self, x):
        x = self.conv_block1(x)
        x = self.conv_block2(x)
        logits = self.classifier(x)
        return logits

model_cnn = MNISTCNN().to(device)
print(model_cnn)

total_params_cnn = sum(p.numel() for p in model_cnn.parameters() if p.requires_grad)
print(f"Total de parâmetros treináveis da CNN: {total_params_cnn:,}")


---
### 4. Treinamento da CNN


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_cnn.parameters(), lr=0.001)

epochs = 5
train_losses_cnn = []
train_accs_cnn = []

for epoch in range(epochs):
    model_cnn.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        outputs = model_cnn(images)
        loss = criterion(outputs, labels)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
    epoch_loss = running_loss / total
    epoch_acc = (correct / total) * 100
    
    train_losses_cnn.append(epoch_loss)
    train_accs_cnn.append(epoch_acc)
    
    print(f"Época [{epoch+1}/{epochs}] | Loss: {epoch_loss:.4f} | Acurácia Treino: {epoch_acc:.2f}%")


---
### 5. Avaliação da CNN no Conjunto de Teste


In [ ]:
model_cnn.eval()
test_loss = 0.0
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model_cnn(images)
        loss = criterion(outputs, labels)
        
        test_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

final_cnn_acc = (correct / total) * 100
print(f"📊 Acurácia Final da CNN no Conjunto de Teste: {final_cnn_acc:.2f}%")


---
### 6. Visualização dos Feature Maps Extraídos pelos Filtros Convolucionais
Vamos inspecionar o que a primeira camada de convolução (`conv_block1[0]`) extraiu de uma imagem do número 7!


In [ ]:
# Selecionar uma imagem de teste
sample_img, sample_label = test_dataset[0]  # Imagem individual (1, 28, 28)
input_tensor = sample_img.unsqueeze(0).to(device)  # Adiciona dimensão de batch (1, 1, 28, 28)

# Passar apenas pela primeira camada convolucional
model_cnn.eval()
with torch.no_grad():
    feature_maps = model_cnn.conv_block1[0](input_tensor) # Forma: (1, 16, 28, 28)

feature_maps_np = feature_maps.squeeze(0).cpu().numpy()

# Visualizar os 16 Feature Maps gerados pelos 16 filtros
fig, axes = plt.subplots(2, 8, figsize=(16, 4))
fig.suptitle(f"16 Feature Maps Gerados pela 1ª Convolução (Dígito {sample_label})", fontsize=14, fontweight='bold')

for i in range(16):
    row = i // 8
    col = i % 8
    axes[row, col].imshow(feature_maps_np[i], cmap='viridis')
    axes[row, col].set_title(f"Filtro {i+1}", fontsize=9)
    axes[row, col].axis('off')

plt.tight_layout()
plt.show()


---
### 7. Resumo da Comparação: MLP vs CNN no MNIST
- **MLP (Parte 1):** Achata a matriz 2D em 784 entradas, perdendo relações locais.
- **CNN (Parte 2):** Opera sobre matrizes 2D, extrai bordas e texturas com compartilhamento de pesos e gera mapas de características visuais com acurácia superior!


---
## 🏆 Conclusão e Resumo Pedagógico da Aula 06

Nesta aula, aprendemos por que o surgimento das CNNs revolucionou a Inteligência Artificial:
- **Preservação Espacial**: O kernel varre a imagem respeitando a vizinhança 2D entre pixels.
- **Compartilhamento de Pesos**: O mesmo detector de borda é aplicado em toda a imagem, reduzindo drasticamente o número de parâmetros.
- **Invariância Translacional**: Graças ao *Max Pooling*, a rede reconhece um dígito mesmo que ele esteja ligeiramente deslocado.

Na **Aula 07**, levaremos as convoluções para o mundo real: trabalharemos com **Imagens RGB Multicanal**, aumento de dados com `torchvision.transforms` e diagnosticaremos doenças foliares no dataset da agritech **PlantVillage**!